# Part II Exact Theory-Baseline Audit — Qwen/Pythia

This notebook **does not rerun any Qwen or Pythia model**. It only reconstructs the synthetic Markov kernels with the original seed rule and recomputes the information-theoretic baselines correctly under the stationary state distribution.

Why this audit exists:

- The original Part II helper `estimate_H_star(P)` averaged row entropies uniformly over the \(V^k\) states.
- The manuscript defines \(H^*=H(X_{t+1}\mid S_t)\), which for a stationary Markov source must be weighted by the stationary state distribution.
- The original \(H_L\) estimator for \(L<k\) also estimated the conditional mixture and evaluated it on the same Monte Carlo sample.
- This notebook removes both issues by computing the stationary distribution numerically and then evaluating \(H^*\) and \(H_L\) directly from the known kernel.

**Critical integrity guard:** before accepting any corrected value, the notebook regenerates each kernel and checks that its original uniform-state entropy matches the `H_star` stored in the supplied Part II result archive. If this check fails, execution stops rather than silently using a different kernel.

**V2 numerical fix:** the stationary solve is performed on CPU float64 after reproducing the kernel on CUDA. This avoids nondeterministic/atomic CUDA accumulation noise in `scatter_add` that can stall around ~1e-10.


In [ ]:
from pathlib import Path
import json, zipfile, shutil, math, hashlib, sys, os
import numpy as np
import pandas as pd
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "Run this notebook on the same CUDA/A100 environment used for Part II. "
        "The original kernels were generated with a CUDA torch.Generator, so CPU regeneration "
        "must not be used for a forensic correction."
    )

DEVICE = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

V = 8
KS = [2, 3, 4]
SEEDS = list(range(10))


## 1. Locate the original Qwen Part-II result ZIP

Upload the original `Part2_Sufficiency_Representation_RESULTS.zip` (the 10-seed run) into the same working directory as this notebook. The code also accepts filenames with `(1)` suffixes.


In [ ]:
cwd = Path.cwd()
candidates = sorted(cwd.glob("Part2_Sufficiency_Representation_RESULTS*.zip"))

# Also check the historical A100 working directory if present.
for p in Path("/home/jamie/data").glob("Part2_Sufficiency_Representation_RESULTS*.zip") if Path("/home/jamie/data").exists() else []:
    if p not in candidates:
        candidates.append(p)

if not candidates:
    raise FileNotFoundError(
        "Could not find Part2_Sufficiency_Representation_RESULTS*.zip. "
        "Upload the 10-seed Qwen result ZIP next to this notebook."
    )

SOURCE_ZIP = candidates[0]
print("Using:", SOURCE_ZIP)

EXTRACT = cwd / "_part2_theory_audit_source"
if EXTRACT.exists():
    shutil.rmtree(EXTRACT)
EXTRACT.mkdir(parents=True)
with zipfile.ZipFile(SOURCE_ZIP, "r") as z:
    z.extractall(EXTRACT)

theory_dir = EXTRACT / "theory_cache"
if not theory_dir.exists():
    raise FileNotFoundError("theory_cache/ was not found in the supplied Qwen result ZIP.")

print("Extracted theory files:", len(list(theory_dir.glob("*.json"))))


## 2. Reconstruct the exact original kernels

This is intentionally the same kernel generator as the Part-II experiment:
\[
P_s=\mathrm{softmax}(\mathrm{randn}(V^k,V)).
\]


In [ ]:
def generate_kernel(V, k, seed, device=DEVICE):
    g = torch.Generator(device=device)
    g.manual_seed(seed + 10000019)
    return torch.softmax(
        torch.randn((V**k, V), generator=g, device=device, dtype=torch.float32),
        dim=-1
    )

@torch.no_grad()
def uniform_row_entropy(P):
    p = P.double()
    return float((-(p * torch.log(p.clamp_min(1e-300))).sum(-1)).mean())


## 3. Exact stationary distribution and conditional entropies

For state index \(s=(x_1,\ldots,x_k)\), the next state after symbol \(y\) is
\[
(x_2,\ldots,x_k,y).
\]
The transition operator is therefore sparse: each state has exactly \(V\) outgoing transitions. We use power iteration in float64 and require a tight stationary residual.


In [ ]:
@torch.no_grad()
def stationary_distribution(P, V, k, tol=1e-13, max_iter=100000):
    """
    Compute the stationary distribution deterministically on CPU in float64.

    Important: P is STILL generated on CUDA above, so the original Part-II
    kernel is reproduced exactly. We only move the already-generated kernel
    to CPU for the stationary solve. This avoids the ~1e-10 numerical floor
    that CUDA scatter_add can exhibit because of parallel atomic accumulation.
    """
    P_np = P.detach().cpu().double().numpy()
    S = V**k

    states = np.arange(S, dtype=np.int64)
    suffix = np.zeros_like(states) if k == 1 else (states % (V**(k-1)))
    y = np.arange(V, dtype=np.int64)
    next_idx = suffix[:, None] * V + y[None, :]

    flat_next = next_idx.reshape(-1)
    pi = np.full(S, 1.0 / S, dtype=np.float64)

    converged = False
    for it in range(max_iter):
        weights = (pi[:, None] * P_np).reshape(-1)
        nxt = np.bincount(flat_next, weights=weights, minlength=S).astype(np.float64)

        # Normalize only against accumulated roundoff.
        nxt /= nxt.sum()

        err = float(np.max(np.abs(nxt - pi)))
        pi = nxt

        if err < tol:
            converged = True
            break

    weights = (pi[:, None] * P_np).reshape(-1)
    chk = np.bincount(flat_next, weights=weights, minlength=S).astype(np.float64)
    chk /= chk.sum()

    residual_max = float(np.max(np.abs(chk - pi)))
    residual_l1 = float(np.sum(np.abs(chk - pi)))

    if not converged:
        raise RuntimeError(
            f"CPU power iteration did not converge for V={V}, k={k}. "
            f"Last max diff={err}"
        )
    if abs(float(pi.sum()) - 1.0) > 1e-12:
        raise RuntimeError("Stationary distribution does not sum to one.")
    if np.min(pi) < -1e-15:
        raise RuntimeError("Negative stationary probability detected.")

    # Return a torch CPU tensor; exact_H_star_and_HL below will move P to CPU too.
    return torch.from_numpy(pi), it + 1, residual_max, residual_l1

@torch.no_grad()
def exact_H_star_and_HL(P, pi, V, k, L):
    """
    Exact entropy calculations on CPU float64 using the already-verified kernel.
    """
    P = P.detach().cpu().double()
    pi = pi.detach().cpu().double()

    row_H = -(P * torch.log(P.clamp_min(1e-300))).sum(-1)
    H_star = torch.sum(pi * row_H)

    if L >= k:
        return float(H_star), float(H_star)

    S = V**k
    state_idx = torch.arange(S, dtype=torch.long)
    suffix_idx = state_idx % (V**L)
    nc = V**L

    mass = torch.zeros(nc, dtype=torch.float64)
    mass.index_add_(0, suffix_idx, pi)

    joint = torch.zeros((nc, V), dtype=torch.float64)
    joint.index_add_(0, suffix_idx, pi[:, None] * P)

    q = joint / mass[:, None].clamp_min(1e-300)
    q_for_state = q[suffix_idx].clamp_min(1e-300)

    H_L = torch.sum(pi * (-(P * torch.log(q_for_state)).sum(-1)))
    return float(H_star), float(H_L)


## 4. Forensic kernel identity check and corrected baselines

The old stored `H_star` is used **only as a fingerprint of the regenerated kernel**. We require agreement before computing the corrected stationary quantities.


In [ ]:
rows = []
fingerprint_failures = []

for k in KS:
    for seed in SEEDS:
        P = generate_kernel(V, k, seed)

        old_uniform_recomputed = uniform_row_entropy(P)

        # Use L=k cache as the cleanest stored fingerprint.
        fp_path = theory_dir / f"V{V}_k{k}_L{k}_seed{seed}.json"
        if not fp_path.exists():
            raise FileNotFoundError(fp_path)

        old = json.load(open(fp_path, "r"))
        old_uniform_stored = float(old["H_star"])
        fp_err = abs(old_uniform_recomputed - old_uniform_stored)

        # Tight enough to verify that the same CUDA-generated kernel was reconstructed.
        if fp_err > 5e-10:
            fingerprint_failures.append((k, seed, old_uniform_stored, old_uniform_recomputed, fp_err))
            continue

        pi, n_iter, residual_max, residual_l1 = stationary_distribution(P, V, k)

        for L in [k-1, k, k+1]:
            Hs_exact, HL_exact = exact_H_star_and_HL(P, pi, V, k, L)

            old_path = theory_dir / f"V{V}_k{k}_L{L}_seed{seed}.json"
            oldj = json.load(open(old_path, "r"))
            old_Hs = float(oldj["H_star"])
            old_HL = float(oldj["H_L"])
            old_gap = float(oldj["info_gap"])

            rows.append({
                "V": V,
                "k": k,
                "L": L,
                "seed": seed,
                "stationary_iterations": n_iter,
                "stationary_residual_max": residual_max,
                "stationary_residual_l1": residual_l1,
                "kernel_fingerprint_abs_error": fp_err,
                "old_Hstar_uniform": old_Hs,
                "corrected_Hstar_stationary": Hs_exact,
                "delta_Hstar_corrected_minus_old": Hs_exact - old_Hs,
                "old_HL": old_HL,
                "corrected_HL_stationary": HL_exact,
                "delta_HL_corrected_minus_old": HL_exact - old_HL,
                "old_info_gap": old_gap,
                "corrected_info_gap": HL_exact - Hs_exact,
            })

if fingerprint_failures:
    print("KERNEL FINGERPRINT FAILURES:")
    for x in fingerprint_failures:
        print(x)
    raise RuntimeError(
        "STOP: regenerated CUDA kernels do not match the original stored fingerprints. "
        "Do not use the corrected values."
    )

audit = pd.DataFrame(rows).sort_values(["k","seed","L"]).reset_index(drop=True)
print("Kernel fingerprint check: PASS for all", len(KS)*len(SEEDS), "kernels")
display(audit.head(15))


## 5. Integrity checks


In [ ]:
assert len(audit) == len(KS) * len(SEEDS) * 3
assert audit["kernel_fingerprint_abs_error"].max() <= 5e-10
assert audit["stationary_residual_max"].max() < 1e-12
assert audit["stationary_residual_l1"].max() < 1e-10

# By definition, once L >= k, the context-limited Bayes entropy must equal H*.
post = audit[audit.L >= audit.k]
assert np.max(np.abs(post.corrected_HL_stationary - post.corrected_Hstar_stationary)) < 1e-13

# Below the boundary, information gap should be non-negative for these dense kernels.
below = audit[audit.L == audit.k - 1]
assert (below.corrected_info_gap >= -1e-12).all()

print("All structural integrity checks: PASS")
print()
print("Maximum |corrected H* - old uniform H*|:",
      audit["delta_Hstar_corrected_minus_old"].abs().max())
print("Maximum |corrected H_L - old H_L|:",
      audit["delta_HL_corrected_minus_old"].abs().max())
print()
print("Corrected mean information gaps at L=k-1:")
display(below.groupby("k")["corrected_info_gap"].agg(["mean","std","min","max"]))


## 6. Export

Upload the resulting ZIP back to ChatGPT. No Qwen/Pythia model inference or probe retraining is required.


In [ ]:
OUT_CSV = cwd / "Part2_THEORY_BASELINE_EXACT_AUDIT.csv"
OUT_SUMMARY = cwd / "Part2_THEORY_BASELINE_EXACT_AUDIT_SUMMARY.json"
OUT_ZIP = cwd / "Part2_THEORY_BASELINE_EXACT_AUDIT_RESULTS.zip"

audit.to_csv(OUT_CSV, index=False)

summary = {
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "kernels_verified": int(len(KS) * len(SEEDS)),
    "rows": int(len(audit)),
    "max_kernel_fingerprint_abs_error": float(audit.kernel_fingerprint_abs_error.max()),
    "max_stationary_residual_max": float(audit.stationary_residual_max.max()),
    "max_stationary_residual_l1": float(audit.stationary_residual_l1.max()),
    "max_abs_Hstar_correction": float(audit.delta_Hstar_corrected_minus_old.abs().max()),
    "max_abs_HL_correction": float(audit.delta_HL_corrected_minus_old.abs().max()),
}
OUT_SUMMARY.write_text(json.dumps(summary, indent=2), encoding="utf-8")

with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(OUT_CSV, OUT_CSV.name)
    z.write(OUT_SUMMARY, OUT_SUMMARY.name)

print("Created:", OUT_ZIP)
print(json.dumps(summary, indent=2))
